# 基于净值的债券基金风险因子归因模型

风险因子：利率(久期+凸性)、信用利差、可转债

In [ ]:
import sys
sys.path.insert(0, r'C:\Users\chenh\.qclaw\workspace\bond_factor_attribution')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from config import setup_chinese_font, OUTPUT_DIR
from source.data_loader import get_fund_nav, get_bond_index_data
from source.factor_builder import build_all_factors
from source.collinearity import diagnose_collinearity
from source.factor_model import FactorRegressionModel, calculate_factor_contribution
from source.plot import plot_factor_exposure, plot_factor_contribution

setup_chinese_font()
print('模块导入成功！')

## 1. 获取数据

In [ ]:
from datetime import datetime, timedelta

fund_code = '110017'
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

# 获取基金净值
fund_nav = get_fund_nav(fund_code, start_date, end_date)
print(f'净值数据: {len(fund_nav)} 条')
fund_nav.head()

In [ ]:
# 获取指数数据
treasury_idx = get_bond_index_data('treasury', start_date, end_date)
print(f'国债指数: {len(treasury_idx)} 条')
treasury_idx.head()

## 2. 构建风险因子

In [ ]:
factors = build_all_factors(treasury_idx)
factors.head()

## 3. 共线性诊断

In [ ]:
factor_names = ['duration_factor', 'convexity_factor']
diagnosis = diagnose_collinearity(factors[factor_names])

print('VIF检验:')
print(diagnosis['vif'])
print('\n建议:')
for rec in diagnosis['recommendation']:
    print(f'  - {rec}')

## 4. 因子回归

In [ ]:
model = FactorRegressionModel()
results = model.fit(fund_nav['daily_return'], factors, factor_names)

print(f'Alpha: {results["alpha"]*100:.4f}%')
print(f'R-squared: {results["r_squared"]:.4f}')
print('\n因子暴露:')
for name, beta in results['factor_exposures'].items():
    print(f'  {name}: {beta:.4f}')

## 5. 可视化

In [ ]:
plot_factor_exposure(results)

In [ ]:
contrib = calculate_factor_contribution(fund_nav['daily_return'], factors, results)
plot_factor_contribution(contrib)